# Transformer Decoder + Positional Encoding — IRB2400 Kinematics

## 이 노트북이 하는 일
Encoder 버전과 **동일한 태스크** — 관절각 시퀀스 `q1~q6` → 다음 포즈 `(x,y,z,yaw,pitch,roll)`.
다른 점은 **Transformer Decoder 블록** 을 쓴다는 것:
- self-attention 에 **causal mask** 를 걸어서 "현재 스텝은 과거만 참조" 하도록 제한

## Encoder 버전과 뭐가 다른가

| 항목            | Encoder 버전      | 이 Decoder 버전          |
|-----------------|-------------------|--------------------------|
| self-attention  | bidirectional (마스크 없음) | **causal** (미래 마스킹) |
| 시간 의존성      | "시퀀스 전체 요약"  | "자기회귀적 요약 (GPT 스타일)" |
| time_steps      | 10                | **20**                    |
| ff_dim          | 128               | 128                       |
| dropout         | 0.1               | 0.1                       |

### 왜 decoder 로도 풀어보는가
- Encoder 는 미래 스텝까지 다 봐서 요약 → 모든 과거 스텝에 대해 동일한 표현
- Decoder 는 causal 이라 각 타임스텝이 **그 시점까지의 정보만 으로 표현** 을 만듦 → 실시간 스트리밍 예측에 더 자연스러운 구조
- 두 방식의 성능을 비교해 보기 위한 실험이다.


---
## [1] 필수 라이브러리

EarlyStopping 을 포함해서 학습 안정성을 챙김.


In [ ]:
# [1] 필수 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Add, LayerNormalization,
    GlobalAveragePooling1D, MultiHeadAttention,
)
from tensorflow.keras.callbacks import EarlyStopping


## [2] Google Drive 마운트 (Colab 전용)


In [ ]:
# [2] Google Drive 마운트 (Colab 전용)
from google.colab import drive
drive.mount('/content/drive')


## [3]~[5] CSV 로딩 / 입·출력 / 정규화 — encoder 버전과 동일


In [ ]:
# [3] CSV 파일 로딩
csv_path = '/content/drive/MyDrive/Colab Notebooks/data/ngv/datasetIRB2400.csv'
df = pd.read_csv(csv_path)
print("데이터 로딩 완료:", df.shape)
print(df.head())

# [4] 입력/출력 정의
X = df[['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']].values
y = df[['x', 'y', 'z', 'yaw', 'pitch', 'roll']].values

# [5] 정규화 — MSE 회귀에서 스케일 맞추기 위한 표준 전처리
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)


## [6] 시퀀스 생성 — `time_steps=20`

encoder baseline 의 10 보다 길게 잡은 이유:
- causal self-attention 은 각 타임스텝이 **그 시점까지의 정보만** 봄
- 따라서 충분한 "과거 문맥" 이 있어야 마지막 스텝이 유용한 정보를 모음
- 20 스텝은 짧지도 길지도 않게 궤적의 가감속 패턴까지 보기에 적절


In [ ]:
# [6] 시퀀스 생성 함수
def create_sequences(X, y, time_steps=20):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i + time_steps])      # (T, 6) 관절각 과거 시퀀스
        ys.append(y[i + time_steps])         # (6,) 다음 시점 포즈
    return np.array(Xs), np.array(ys)

time_steps = 20
X_seq, y_seq = create_sequences(X_scaled, y_scaled, time_steps)
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


## [7] Positional Encoding

Encoder 버전과 동일한 sinusoidal PE.
- self-attention 은 순서를 모르므로 위치 정보가 필요
- sin/cos 방식은 학습 파라미터 없음 → 작은 데이터셋에서 일반화 유리


In [ ]:
# [7] Positional Encoding Layer — sinusoidal (학습 파라미터 0개)
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        d_model = inputs.shape[-1]

        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        i   = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)

        angle_rates = 1 / tf.pow(10000.0, (2 * (i // 2)) / tf.cast(d_model, tf.float32))
        angle_rads  = pos * angle_rates

        sines   = tf.sin(angle_rads[:, 0::2])
        cosines = tf.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = pos_encoding[tf.newaxis, ...]
        return inputs + pos_encoding


## [8] Causal Self-Attention — 이 노트북의 핵심

### 왜 "causal" (인과) 마스크를 거는가
Transformer Decoder 의 정의상 특성. 각 토큰(시점) 이 자기 자신과 **과거 시점** 만 참조하도록 제한.

### 구현: 하삼각 마스크
```
T=4 일 때 mask:
1 0 0 0     ← 시점 0 은 자기만 봄
1 1 0 0     ← 시점 1 은 0,1 봄
1 1 1 0     ← 시점 2 는 0,1,2 봄
1 1 1 1     ← 시점 3 은 모두 봄
```
`tf.linalg.band_part(ones, -1, 0)` 가 정확히 이 하삼각 행렬을 만든다.

### attention_axes=1 의 의미 (원본 코드)
`MultiHeadAttention(attention_axes=1)` 은 축 1 (시퀀스 축) 에만 attention 을 걸도록 지정.
여기서는 입력이 `(B, T, d)` 이라 기본 동작과 동일하지만 명시적 지정.

### 주의사항
원본 코드에서 `causal_mask` 를 `(1, 1, T, T)` shape 로 broadcast 하지만 `attention_axes=1` 과 함께 쓰면 내부 처리가 약간 달라질 수 있음. 실전에서는 단순히 `MultiHeadAttention(..., use_causal_mask=True)` (Keras 3) 또는 `attention_mask` 로 명시 전달하는 방식이 더 안전.


In [ ]:
# [8] Causal Self-Attention 레이어
class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, head_size, num_heads, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
            attention_axes=1,       # 시퀀스 축 지정 (명시적)
        )

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        # 하삼각 mask: 1=볼 수 있음, 0=마스킹
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        causal_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]  # (1, 1, T, T) — head 축 broadcast
        return self.attn(inputs, inputs, attention_mask=causal_mask)


## [9] Decoder Block

### 구조 (Post-Norm)
```
x ─ CausalSelfAttn ─ +x ─ LN ─ FFN ─ +x ─ LN ─ out
```

### Encoder 버전과 차이
- encoder 는 Pre-Norm (LN → Attn → Add), 여기는 **Post-Norm** (Attn → Add → LN)
- 얕은 네트워크(2블록) 에서는 둘 다 잘 동작. 깊어지면 Pre-Norm 이 안정적.

### FFN 중간 Dropout
표현력 큰 FFN hidden 뒤에 Dropout 을 넣어 과적합 완화.
Self-attention 자체는 이미 내부 `dropout` 옵션으로 정규화됨.


In [ ]:
# [9] Decoder Block
def decoder_block(inputs, head_size=32, num_heads=4, ff_dim=128, dropout=0.1):
    # ---- 1) Causal Self-Attention (미래 차단) ----
    attn_output = CausalSelfAttention(head_size, num_heads, dropout)(inputs)
    x = Add()([inputs, attn_output])
    x = LayerNormalization(epsilon=1e-6)(x)     # Post-Norm: Add 다음에 LN

    # ---- 2) Feed-Forward ----
    ff = Dense(ff_dim, activation='relu')(x)
    ff = Dropout(dropout)(ff)                    # FFN hidden 에 dropout
    ff = Dense(inputs.shape[-1])(ff)             # 원래 차원으로 복원
    x = Add()([x, ff])
    return LayerNormalization(epsilon=1e-6)(x)


## [10] 모델 구성

### 하이퍼파라미터 선택
- `head_size=32, num_heads=4` → 총 attention dim 128
- `ff_dim=128` → FFN hidden
- `dropout=0.1` → 적당한 정규화
- Decoder 2블록 + GlobalAveragePooling + Dense(64) + Dropout(0.2) + Dense(6)

### 왜 GlobalAveragePooling 을 쓰는가 (마지막 토큰이 아니라)
Causal decoder 는 "마지막 토큰만 쓰는 게" 언어 모델 관례지만, 이 회귀 태스크에서는:
- 마지막 토큰 하나만 쓰면 그 한 스텝의 노이즈에 민감
- 평균 풀링은 전체 시퀀스의 안정적 요약 → 회귀 품질 ↑


In [ ]:
# [10] 모델 구성
input_shape = (time_steps, X_train.shape[2])            # (20, 6)
inputs = Input(shape=input_shape)
x = PositionalEncoding()(inputs)

# Decoder 2 블록 — 각 스텝은 자기 이전까지만 참조
x = decoder_block(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.1)
x = decoder_block(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.1)

# 회귀 head
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(6)(x)                                    # 선형 출력 (회귀)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


## [11] 학습 — EarlyStopping

`patience=5, restore_best_weights=True`
- 너무 짧은 patience: 초기 불안정기를 못 넘기고 중단될 수 있음
- 너무 긴 patience: 과적합 기간 동안에도 학습 계속 → 의미 없음
- 5 는 보통 로버스트한 중간값
- `restore_best_weights` 가 **핵심** — 최종 평가는 최고 성능 시점 가중치로


In [ ]:
# [11] 학습
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)


## [12]~[13] 예측 · 역정규화 · MAE

정규화된 공간에서 예측 → `scaler_y.inverse_transform` 으로 실제 단위 복원.
정답도 같이 역변환해서 같은 좌표계에서 오차 계산.


In [ ]:
# [12] 예측 및 역정규화
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

# [13] MAE 계산
mae = mean_absolute_error(y_true, y_pred)
print(f"\n실제 단위 기준 MAE: {mae:.4f}")


## [14]~[15] 시각화

1. Loss 곡선 — 과적합 진단
2. x 좌표 True vs Pred — 시간 패턴 진단


In [ ]:
# [14] Loss 곡선
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Loss Curve'); plt.legend(); plt.grid(True)
plt.show()

# [15] x 좌표 예측 vs 실제
plt.plot(y_true[:, 0], label='True x')
plt.plot(y_pred[:, 0], label='Predicted x')
plt.title('Predicted vs True - x')
plt.xlabel('Sample Index'); plt.ylabel('x')
plt.legend(); plt.grid(True)
plt.show()
